### 4.3 Ensemble — inverse-RMSE blend, and the flagship demo

$$\hat y_{ens,h} = w_h\,\hat y_{SARIMA,h} + (1-w_h)\,\hat y_{EN,h}$$

$$w_h \propto \frac{1}{\mathrm{RMSE}_{SARIMA,h}}$$

Whichever family won at *that specific horizon* in the walk-forward comparison gets more
say — the weight can and does change from horizon 1 to horizon 8. This is also the
project's reportable forecast, so it's the one worth actually watching get built: the cell
below truncates to the project's pinned forecast origin, draws 1,000 Monte-Carlo paths from
the fitted SARIMA + Elastic Net components, combines them horizon-by-horizon, and animates
a sample of them fanning out into the **raw, uncalibrated** 80%/50% simulated interval —
press play. This is *not* the same interval `forecast.csv`/the live API actually serve —
see the width comparison after the animation for exactly how much narrower it is.

*Source: `src/models/ensemble.py`*

#### Explained, step by step

- This is a weighted average of the two accuracy-family forecasts at a given horizon $h$. If $w_h=0.7$, the ensemble forecast is 70% SARIMA's number and 30% Elastic Net's number, for that horizon specifically.
- The weight is set by each model's own track record at that horizon: $w_h$ is proportional to $1/\mathrm{RMSE}_{SARIMA,h}$ — the worse SARIMA's historical RMSE at horizon $h$, the smaller its share (small RMSE $\to$ large $1/\mathrm{RMSE}$ $\to$ more weight; large RMSE $\to$ small $1/\mathrm{RMSE}$ $\to$ less weight). In practice this is normalized so the two inverse-RMSEs sum to 1: $w_h = \dfrac{1/\mathrm{RMSE}_{SARIMA,h}}{1/\mathrm{RMSE}_{SARIMA,h} + 1/\mathrm{RMSE}_{EN,h}}$. Because the weight is computed *per horizon*, SARIMA can dominate at horizon 1 while Elastic Net dominates at horizon 6, if that's what the walk-forward accuracy says.
- Intuition: the same logic as weighting two independent instrument readings by how noisy each one is — trust the more reliable source more, with the weighting adapting automatically rather than being hand-picked.

In [42]:
from src.models import ensemble, svar
from src.models.elastic_net import ELASTIC_NET_FEATURE_COLUMNS, TARGET_COLUMN, load_elastic_net_feature_frame
from src.models.evaluation import load_target_series
from src.models.interval_coverage import DEFAULT_SEED
from src.models.simulation_fan import DEFAULT_N_SIMS

ORIGIN = svar.FORECAST_ORIGIN_PIN  # 2025Q4 -- the same pin every other report in this project shares

series = load_target_series(CURATED, target_column=TARGET_COLUMN)
features = load_elastic_net_feature_frame(CURATED, feature_columns=ELASTIC_NET_FEATURE_COLUMNS)
frame = pd.concat([series.rename(TARGET_COLUMN), features], axis=1).dropna().sort_index()

print(f"curated data actually runs to cpi_yoy={series.index[-1]}, macro block={frame.index[-1]} "
      f"-- truncating both to the shared pin {ORIGIN} instead of drifting forward with them")
series = series.loc[:ORIGIN]
frame = frame.loc[:ORIGIN]
assert series.index[-1] == frame.index[-1] == ORIGIN

paths = ensemble.simulate_ensemble_paths(
    frame, steps=8, n_sims=DEFAULT_N_SIMS, seed=DEFAULT_SEED,
    target_column=TARGET_COLUMN, sarima_series=series,
)
forecast_origin = str(frame.index[-1])
quarters = [str(frame.index[-1] + h) for h in range(1, 9)]
p10, p25, med, p75, p90 = np.percentile(paths, [10, 25, 50, 75, 90], axis=0)
print(f"origin {forecast_origin}: median h1={med[0]:.2f}%  h8={med[-1]:.2f}%  "
      f"(80% band at h8: [{p10[-1]:.2f}, {p90[-1]:.2f}])")

# Reproducibility check -- same pin, same seed, same method as the checked-in
# report, so these should match almost exactly (small drift only if ABS has
# since revised a historical print between when that report was generated and now).
_checked_in = pd.read_csv(PROJECT_ROOT / "reports/simulation_fan_ensemble.csv")
_ref = _checked_in.iloc[0]
print(f"reproducibility -- reports/simulation_fan_ensemble.csv h1 median={_ref['median']:.2f}% "
      f"vs. this notebook's h1 median={med[0]:.2f}%  (confirms the pipeline, not the point forecast below)")

# This is a *different* comparison: the deterministic point forecast actually served
# by the API / shown in Tableau and Streamlit used to NOT be the median of
# these simulated paths -- it comes from ensemble.combine_point_forecasts(),
# which weighted-averages each component's own single point prediction
# directly, while the median above weighted-averages 1,000 *paired* Monte
# Carlo draws and only then takes the 50th percentile -- not the same
# arithmetic, since median doesn't distribute over a weighted sum of two
# independently-simulated components. As of 2026-09-14,
# ensemble.recenter_paths_to_median() shifts the combined draws per horizon
# by exactly this gap, so the two now agree exactly (up to floating point) --
# the print below should show ~0.000pp at every horizon.
_reported = pd.read_csv(PROJECT_ROOT / "reports/tableau/forecast.csv")
_reported = _reported[(_reported["model_family"] == "ensemble") & (_reported["target"] == "Headline")]
_reported_vals = _reported.sort_values("horizon")["forecast"].to_numpy()
_gap_median = _reported_vals - med
print(f"reported vs. simulated median -- h1 gap={_gap_median[0]:+.3f}pp, h8 gap={_gap_median[-1]:+.3f}pp "
      f"(recentered onto the point forecast by construction, since 2026-09-14)")

# The simulated *mean* is deliberately left off this recentering, and still
# shows a gap. Mean, unlike median, IS linear, so mean(w*A + (1-w)*B) =
# w*mean(A) + (1-w)*mean(B) exactly -- but recentering targets the MEDIAN
# (matching this project's existing median-centering convention), and a pure
# additive shift moves the whole distribution, mean included, without
# changing its shape. CPI's residual pool is right-skewed (occasional sharp
# upside prints -- GST, COVID reopening, 2021-23 inflation surge -- with no
# equally sharp downside counterpart), so the mean still sits above both the
# now-exact median and the reported point forecast at every horizon.
sim_mean = paths.mean(axis=0)
_gap_mean = _reported_vals - sim_mean
print(f"reported vs. simulated mean   -- h1 gap={_gap_mean[0]:+.3f}pp, h8 gap={_gap_mean[-1]:+.3f}pp "
      f"(further off than the median at every horizon -- consistent with right-skewed residuals)")

curated data actually runs to cpi_yoy=2026Q2, macro block=2026Q1 -- truncating both to the shared pin 2025Q4 instead of drifting forward with them


origin 2025Q4: median h1=3.15%  h8=2.89%  (80% band at h8: [1.70, 4.47])
reproducibility -- reports/simulation_fan_ensemble.csv h1 median=3.15% vs. this notebook's h1 median=3.15%  (confirms the pipeline, not the point forecast below)
reported vs. simulated median -- h1 gap=+0.000pp, h8 gap=+0.000pp (recentered onto the point forecast by construction, since 2026-09-14)
reported vs. simulated mean   -- h1 gap=-0.059pp, h8 gap=-0.124pp (further off than the median at every horizon -- consistent with right-skewed residuals)


**Static snapshot of the animation above.** The live notebook cell drew 1,000 Monte-Carlo paths and animated them fanning out one by one; a Jupyter Book page is static HTML, so instead of embedding a ~15MB animation this page shows the same pinned draws revealed at four checkpoints (1, 10, 40, and all 120 sampled paths):

![Ensemble Monte-Carlo draws revealed at four checkpoints](../_static/ensemble_reveal.png)

```{tip}
To actually drag the slider or press **▶ Play** and watch the paths reveal continuously, run `streamlit run app/streamlit_app.py` from the project root and open **4.3 Ensemble**.
```

In [44]:
# The band above is the RAW p10-p90 spread of these 1,000 draws -- not the
# calibrated interval actually served by forecast.csv / the live API.
# interval_calibration.py found the raw simulated interval under-covers (fewer
# than 80% of real outcomes historically fell inside it), so serving widens each
# side of the interval around the point forecast by a rolling scale factor learned
# from the last 12 quarters of observed forecast errors.
# That scaling isn't reproduced here -- this animation is illustrating what the
# simulation itself looks like, not the calibrated product.
_raw_width = p90 - p10
_cal = pd.read_csv(PROJECT_ROOT / "reports/tableau/forecast.csv")
_cal = _cal[(_cal["model_family"] == "ensemble") & (_cal["target"] == "Headline")].sort_values("horizon")
_calibrated_width = (_cal["interval_upper"] - _cal["interval_lower"]).to_numpy()
_scale = _calibrated_width / _raw_width
print(f"raw vs. calibrated interval width -- h1: {_raw_width[0]:.2f} vs {_calibrated_width[0]:.2f} "
      f"({_scale[0]:.2f}x)  |  h8: {_raw_width[-1]:.2f} vs {_calibrated_width[-1]:.2f} ({_scale[-1]:.2f}x)")

raw vs. calibrated interval width -- h1: 1.32 vs 4.77 (3.61x)  |  h8: 2.77 vs 10.02 (3.61x)
